## 5. ML Modeling & Training

### Task Context

This section implements a **one-class classification** approach to detect hidden SMEs among consumer cards.

Key design decisions driven by the MDQ problem statement:

| Decision | Rationale |
|---|---|
| Unit of prediction = card | Hidden SME is a card-holder property, not a per-transaction property |
| Primary metric = AUC-ROC | Dataset is imbalanced (~24% business); accuracy would be misleading |
| Train on business cards | All 25 000 business cards are labeled ground truth (class = 1) |
| Consumer cards = negatives | 80 000 consumer cards treated as class = 0 for supervised learning |
| Cross-validation on Dataset X | No external test set with true hidden-SME labels; CV is the only self-evaluation tool |

> **Note:** The final submission format expected by Mastercard is `card_number, score` — one row per consumer card.
> AUC-ROC is the official evaluation metric (not accuracy). See the FAQ section below.

---

### FAQ from Mastercard (MDQ)

**Q: Why not use accuracy?**  
Class 'hidden business' is a small fraction of consumer cards. A model predicting everyone as 'consumer' can reach 70–90% accuracy while finding zero hidden businesses. AUC-ROC evaluates ranking quality independently of threshold.

**Q: How will results be evaluated?**  
Mastercard holds the true labels for hidden SMEs in the consumer dataset. Submissions are scored by comparing `card_number, score` predictions to those ground-truth labels using AUC-ROC.

**Q: Should the model label individual transactions?**  
No. Aggregate all transactions per card into behavioral features, then output one score per `card_number`.

**Q: Can external datasets be used?**  
No. Only the provided parquet files.

### 5.1 Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score, accuracy_score
)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import joblib

RANDOM_STATE = 42
N_FOLDS     = 5
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid']      = True

### 5.2 Load card-level feature tables

In [ ]:
business_card_features = pd.read_parquet('business_card_features.parquet')
consumer_card_features  = pd.read_parquet('consumer_card_features.parquet')

print('Business card features:', business_card_features.shape)
print('Consumer card features: ', consumer_card_features.shape)

### 5.3 Build the labeled training matrix

All 25 000 business cards are **positive examples** (label = 1).  
All 80 000 consumer cards are **negative examples** (label = 0) for supervised training.  
This is the standard proxy-label approach for one-class detection when no ground-truth for the negative class is available.


In [ ]:
EXCLUDE_COLS = ['card_number', 'first_transaction', 'last_transaction']

FEATURE_COLS = [
    c for c in business_card_features.columns
    if c not in EXCLUDE_COLS
]

X_business = business_card_features[FEATURE_COLS].copy()
X_consumer  = consumer_card_features[FEATURE_COLS].copy()

X_train_full = pd.concat([X_business, X_consumer], ignore_index=True)
y_train_full = np.array([1] * len(X_business) + [0] * len(X_consumer))

print('Training matrix shape:  ', X_train_full.shape)
print('Class distribution:  1 (business):', y_train_full.sum(),
      ' | 0 (consumer):', (y_train_full == 0).sum())
print('Positive class ratio: ', round(y_train_full.mean(), 4))

### 5.4 Train/test split

Stratified 80/20 split preserves the class ratio in both halves.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_train_full, y_train_full,
    test_size=0.20,
    stratify=y_train_full,
    random_state=RANDOM_STATE
)

print('Train size:', X_train.shape, '  Positives:', y_train.sum())
print('Test  size:', X_test.shape,  '  Positives:', y_test.sum())

### 5.5 Cross-validation helper

All models are evaluated using **Stratified 5-Fold CV** on the training set.  
AUC-ROC is the primary metric; accuracy is logged for reference only.


In [ ]:
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

cv_results = {}  # {model_name: {'auc': [...], 'acc': [...]}}

def run_cv(name, model, X, y):
    auc_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    acc_scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_results[name] = {'auc': auc_scores, 'acc': acc_scores}
    print(f'{name:30s}  AUC: {auc_scores.mean():.4f} ± {auc_scores.std():.4f}'
          f'  |  Acc: {acc_scores.mean():.4f} ± {acc_scores.std():.4f}')
    return model

### 5.6 Baseline models

#### Logistic Regression
Linear baseline. Requires feature scaling — wrapped in a `Pipeline`.


In [ ]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ))
])

run_cv('Logistic Regression', lr_pipeline, X_train, y_train)
lr_pipeline.fit(X_train, y_train)

#### Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=5,
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_STATE
)

run_cv('Random Forest', rf_model, X_train, y_train)
rf_model.fit(X_train, y_train)

### 5.7 Advanced models

#### XGBoost
`scale_pos_weight` compensates for class imbalance (ratio of negatives to positives).


In [ ]:
scale_pos_weight = (y_train == 0).sum() / y_train.sum()
print(f'scale_pos_weight: {scale_pos_weight:.4f}')

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    early_stopping_rounds=30,
    random_state=RANDOM_STATE,
    verbosity=0
)

# Early stopping requires a separate eval set — fit outside CV first
from sklearn.model_selection import train_test_split as tts
X_tr, X_val, y_tr, y_val = tts(X_train, y_train, test_size=0.15,
                                 stratify=y_train, random_state=RANDOM_STATE)
xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)
print(f'XGBoost best iteration: {xgb_model.best_iteration}')

# CV with best n_estimators
xgb_cv = XGBClassifier(
    n_estimators=xgb_model.best_iteration,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    verbosity=0
)
run_cv('XGBoost', xgb_cv, X_train, y_train)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

#### CatBoost
Handles the slight class imbalance natively via `auto_class_weights`.


In [ ]:
cat_model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    auto_class_weights='Balanced',
    eval_metric='AUC',
    early_stopping_rounds=30,
    random_seed=RANDOM_STATE,
    verbose=False
)

cat_model.fit(
    X_tr, y_tr,
    eval_set=(X_val, y_val),
    verbose=False
)
print(f'CatBoost best iteration: {cat_model.best_iteration_}')

cat_cv = CatBoostClassifier(
    iterations=cat_model.best_iteration_,
    depth=6,
    learning_rate=0.05,
    auto_class_weights='Balanced',
    random_seed=RANDOM_STATE,
    verbose=False
)
run_cv('CatBoost', cat_cv, X_train, y_train)
cat_model.fit(X_train, y_train, verbose=False)

### 5.8 Hyperparameter tuning — XGBoost (RandomizedSearchCV)

In [ ]:
param_dist = {
    'n_estimators':     [200, 300, 400, 500],
    'max_depth':        [4, 5, 6, 7, 8],
    'learning_rate':    [0.01, 0.03, 0.05, 0.1],
    'subsample':        [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        verbosity=0,
        eval_metric='auc'
    ),
    param_distributions=param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0
)

xgb_search.fit(X_train, y_train)
print('Best XGBoost params:', xgb_search.best_params_)
print('Best CV AUC:        ', round(xgb_search.best_score_, 4))

best_xgb = xgb_search.best_estimator_
run_cv('XGBoost (tuned)', best_xgb, X_train, y_train)

### 5.9 Feature importance — top features

Random Forest and XGBoost both provide impurity-based feature importances.  
Features with near-zero importance across both models are candidates for removal.


In [ ]:
feat_imp_rf  = pd.Series(rf_model.feature_importances_,  index=FEATURE_COLS).sort_values(ascending=False)
feat_imp_xgb = pd.Series(best_xgb.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

feat_imp_rf.head(20).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Random Forest — Top 20 Features')
axes[0].invert_yaxis()

feat_imp_xgb.head(20).plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('XGBoost (tuned) — Top 20 Features')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# Keep features that rank in top-30 in at least one model
top_rf  = set(feat_imp_rf.head(30).index)
top_xgb = set(feat_imp_xgb.head(30).index)
SELECTED_FEATURES = sorted(top_rf | top_xgb)
print(f'Selected {len(SELECTED_FEATURES)} features (union of top-30 from RF and XGB)')

### 5.10 Final model selection

Retrain CatBoost and XGBoost (tuned) on the **selected feature subset** and pick the winner by test AUC.


In [ ]:
X_train_sel = X_train[SELECTED_FEATURES]
X_test_sel  = X_test[SELECTED_FEATURES]

# Final CatBoost
final_cat = CatBoostClassifier(
    iterations=cat_model.best_iteration_,
    depth=6,
    learning_rate=0.05,
    auto_class_weights='Balanced',
    random_seed=RANDOM_STATE,
    verbose=False
)
final_cat.fit(X_train_sel, y_train, verbose=False)

# Final XGBoost
final_xgb = XGBClassifier(**xgb_search.best_params_,
                           scale_pos_weight=scale_pos_weight,
                           random_state=RANDOM_STATE,
                           verbosity=0)
final_xgb.fit(X_train_sel, y_train)

auc_cat = roc_auc_score(y_test, final_cat.predict_proba(X_test_sel)[:, 1])
auc_xgb = roc_auc_score(y_test, final_xgb.predict_proba(X_test_sel)[:, 1])

print(f'CatBoost test AUC: {auc_cat:.4f}')
print(f'XGBoost  test AUC: {auc_xgb:.4f}')

FINAL_MODEL      = final_cat if auc_cat >= auc_xgb else final_xgb
FINAL_MODEL_NAME = 'CatBoost' if auc_cat >= auc_xgb else 'XGBoost'
print(f'\n>>> Selected final model: {FINAL_MODEL_NAME}')